# RAG Pipeline - From Data Ingestion to Vector DB Pipeline

In [123]:
# Import importent libraries
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [124]:
## Read all the PDFs from directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n processing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" ✅ Loaded {len(documents)} pages")

        except Exception as e:
            print(f" ❌ Error while loading documents: {e}")


    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

## Process all the PDF in data Directory
all_pdf_documents = process_all_pdfs("../data")

Found 10 PDF files to process

 processing: System_Design_Cloud_Interview_Prep.md.pdf
 ✅ Loaded 53 pages

 processing: JVM_Internals_Collections_Interview_Prep.md.pdf
 ✅ Loaded 46 pages

 processing: Microservices_Interview_Prep_with_Code.md.pdf
 ✅ Loaded 55 pages

 processing: spring-ai-interview-guide.md.pdf
 ✅ Loaded 13 pages

 processing: Java_Multithreading_Interview_Prep_with_Code.md.pdf
 ✅ Loaded 58 pages

 processing: Java8_Features_Interview_Prep.md.pdf
 ✅ Loaded 84 pages

 processing: SQL_Database_Internals_Interview_Prep.md.pdf
 ✅ Loaded 50 pages

 processing: Java_Multithreading_Basic_To_Advanced.md.pdf
 ✅ Loaded 110 pages

 processing: SpringBoot_SpringCloud_Interview_Prep.md.pdf
 ✅ Loaded 58 pages

 processing: ReactJS_Interview_Prep.md.pdf
 ✅ Loaded 57 pages

 Total documents loaded: 584


In [125]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-30T04:57:48+00:00', 'source': '../data/pdf_file/System_Design_Cloud_Interview_Prep.md.pdf', 'file_path': '../data/pdf_file/System_Design_Cloud_Interview_Prep.md.pdf', 'total_pages': 53, 'format': 'PDF 1.4', 'title': 'System_Design_Cloud_Interview_Prep.md', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-30T04:57:48+00:00', 'trapped': '', 'modDate': "D:20260630045748+00'00'", 'creationDate': "D:20260630045748+00'00'", 'page': 0, 'source_file': 'System_Design_Cloud_Interview_Prep.md.pdf', 'file_type': 'pdf'}, page_content='System Design (Cloud) — Complete Interview Preparation\nGuide (with Code Examples)\nFor: Experienced Java Developer (8+ years)\nEvery question below has a detailed explanation and a code example — Java\nimplementations for algorithmic pieces (rate li

# Text Spliting to get into chunks

In [126]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG Performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    ## Show example of a chunks
    if split_docs:
        print(f"\nExample chunks:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [127]:
chunks = split_documents(all_pdf_documents)

Split 584 documents into 1102 chunks

Example chunks:
Content: System Design (Cloud) — Complete Interview Preparation
Guide (with Code Examples)
For: Experienced Java Developer (8+ years)
Every question below has a detailed explanation and a code example — Java
i...
Metadata: {'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-30T04:57:48+00:00', 'source': '../data/pdf_file/System_Design_Cloud_Interview_Prep.md.pdf', 'file_path': '../data/pdf_file/System_Design_Cloud_Interview_Prep.md.pdf', 'total_pages': 53, 'format': 'PDF 1.4', 'title': 'System_Design_Cloud_Interview_Prep.md', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-30T04:57:48+00:00', 'trapped': '', 'modDate': "D:20260630045748+00'00'", 'creationDate': "D:20260630045748+00'00'", 'page': 0, 'source_file': 'System_Design_Cloud_Interview_Prep.md.pdf', 'file_type': 'pdf'}


# Embiding and vectorStorDB

## Importing importent libraries

In [128]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import ollama

In [129]:
class EmbeddingManager:
    """Handles documents embedding generation using Sentencetransformer"""
    def __init__(self, model_name: str = "qwen3-embedding"):
        """
        Initialize the embedding manager
        Args:
            model_name: Ollama model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error while loading model: {self.model_name} : {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model Not loaded")
        return self.model.get_embedding_dimension()


In [130]:
class OllamaEmbeddingManager:
    """Handels document embedding generation using Ollama"""

    def __init__(self, model_name: str = "qwen3-embedding", batch_size: int = 50):
        """
        Initialize the embedding manager
        
        Args:
            model_name: Ollama model name for sentence embedding
        """

        self.model_name = model_name
        self.model = None
        self.batch_size = batch_size
        self._embedding_dimension = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = ollama.embeddings(
                model = self.model_name,
                prompt = "Test connection"
            )

            self._embedding_dimension = len(self.model["embedding"])
            print(f" ✅ connected to ollama model: {self.model_name}")
            print(f" ✅ Embedding Dimension: {self._embedding_dimension}")
        except Exception as e:
            print(f"❌ Error connecting to Ollama: {e}")
            print(f"Make sure Ollama is running a model: {self.model_name}")
            raise

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts

        Args:
            texts: List of texts string to embed

        Returns:
        numpy array of embedding with shape (len(texts), embedding_dimension)
        """
        if not texts:
            raise ValueError("Texts list is empty")

        print(f"Generating embedding for {len(texts)} texts...")
        embeddings = []

        # Process in batches to avoid overwhelming the system
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            batch_embeddings = []

            for text in batch:
                try:
                    response = ollama.embeddings(
                        model=self.model_name,
                        prompt=text
                    )
                    batch_embeddings.append(response["embedding"])
                except Exception as e:
                    print(f"❌ Error embedding text: {e}")
                    raise

            embeddings.extend(batch_embeddings)

            # Progress indicator
            processed = min(i+self.batch_size, len(texts))
            print(f" Progress: {processed}/{len(texts)} texts processed")

        embeddings_array = np.array(embeddings)
        print(f"✅ Generated embeddings with shape: {embeddings_array.shape}")
        return embeddings_array

    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if self._embedding_dimension is None:
            raise ValueError("Embedding model not initialized properly")
        return self._embedding_dimension

ollama_embedding_manager = OllamaEmbeddingManager()

Loading embedding model: qwen3-embedding
 ✅ connected to ollama model: qwen3-embedding
 ✅ Embedding Dimension: 4096


In [131]:
ollama_embedding_manager

# Vectore Store

In [132]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents_2", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"desctiption":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialize, collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializeing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: list of langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")

        # Prefare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ## Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            ## Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            ## Document content
            documents_text.append(doc.page_content)

            ## Embedding
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector store initialize, collection: pdf_documents_2
Existing documents in collection: 0


In [133]:
chunks

[Document(metadata={'producer': 'Skia/PDF m149', 'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36', 'creationdate': '2026-06-30T04:57:48+00:00', 'source': '../data/pdf_file/System_Design_Cloud_Interview_Prep.md.pdf', 'file_path': '../data/pdf_file/System_Design_Cloud_Interview_Prep.md.pdf', 'total_pages': 53, 'format': 'PDF 1.4', 'title': 'System_Design_Cloud_Interview_Prep.md', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-30T04:57:48+00:00', 'trapped': '', 'modDate': "D:20260630045748+00'00'", 'creationDate': "D:20260630045748+00'00'", 'page': 0, 'source_file': 'System_Design_Cloud_Interview_Prep.md.pdf', 'file_type': 'pdf'}, page_content='System Design (Cloud) — Complete Interview Preparation\nGuide (with Code Examples)\nFor: Experienced Java Developer (8+ years)\nEvery question below has a detailed explanation and a code example — Java\nimplementations for algorithmic pieces (rate li

In [134]:
## Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

['System Design (Cloud) — Complete Interview Preparation\nGuide (with Code Examples)\nFor: Experienced Java Developer (8+ years)\nEvery question below has a detailed explanation and a code example — Java\nimplementations for algorithmic pieces (rate limiters, consistent hashing, LRU cache), and\npseudocode/schemas/configs for architectural patterns.\nTABLE OF CONTENTS\n1. System Design Fundamentals\n2. Capacity Estimation & Back-of-Envelope Calculations\n3. Load Balancing & Global Traffic Management\n4. Caching Strategies & Invalidation\n5. Database Scaling — Replication & Sharding\n6. Consistent Hashing\n7. CAP Theorem & PACELC\n8. Consensus & Leader Election\n9. Message Queues & Stream Processing\n10. Rate Limiting Algorithms\n11. CDN & Edge Architecture\n12. Distributed Caching & Eviction Policies\n13. Object/Blob Storage Design\n14. Distributed File Systems\n15. API Design for Scale\n16. Cloud Architecture Patterns\n17. High Availability & Fault Tolerance\n18. Distributed Locks & C

In [135]:
## Generate the embeddings
embeddings = ollama_embedding_manager.generate_embedding(texts)

## Store in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embedding for 1102 texts...
 Progress: 50/1102 texts processed
 Progress: 100/1102 texts processed
 Progress: 150/1102 texts processed
 Progress: 200/1102 texts processed
 Progress: 250/1102 texts processed
 Progress: 300/1102 texts processed
 Progress: 350/1102 texts processed
 Progress: 400/1102 texts processed
 Progress: 450/1102 texts processed
 Progress: 500/1102 texts processed
 Progress: 550/1102 texts processed
 Progress: 600/1102 texts processed
 Progress: 650/1102 texts processed
 Progress: 700/1102 texts processed
 Progress: 750/1102 texts processed
 Progress: 800/1102 texts processed
 Progress: 850/1102 texts processed
 Progress: 900/1102 texts processed
 Progress: 950/1102 texts processed
 Progress: 1000/1102 texts processed
 Progress: 1050/1102 texts processed
 Progress: 1100/1102 texts processed
 Progress: 1102/1102 texts processed
✅ Generated embeddings with shape: (1102, 4096)
Adding 1102 documents to vector store...
Successfully added 1102 documents to vect

# RAG: Retrival Pipeline from VectorStore

In [136]:
class RAGRetriever:
    """Handles query-based retrival. from the vector store"""

    def __init__(self, vector_store: VectorStore, ollama_embedding_manager: OllamaEmbeddingManager):
        """
        Initialize the retirever

        Args:
            vector_store: Vector store containing documents embeddings
            ollama_embedding_manager: Manager for generationg for query embeddings
        """
        self.vector_store = vector_store
        self.ollama_embedding_manager = ollama_embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Reterieve relevent documents for the query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrived documents and metadata
        """
        print(f"Reterieving documents for query: '{query}'")
        print(f"Top k: {top_k}, score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.ollama_embedding_manager.generate_embedding([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank': i + 1
                            }
                        )

                print(f"Reterieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No document found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrival: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, ollama_embedding_manager)

In [137]:
rag_retriever

In [138]:
result_docs = rag_retriever.retrieve("What\'s the difference between latency and throughput")
result_docs
for res in result_docs:
    print(res['content'])


Reterieving documents for query: 'What's the difference between latency and throughput'
Top k: 5, score threshold: 0.0
Generating embedding for 1 texts...
 Progress: 1/1 texts processed
✅ Generated embeddings with shape: (1, 4096)
Reterieved 5 documents (after filtering)
System.out.println("Parallel sum:   " + sumPar + " in " + (System.curre
// Parallel is typically FASTER here due to multicore CPU utilization f
// NOT BENEFICIAL (and potentially WRONG): mutable shared state with pa
List<Integer> small = Arrays.asList(1, 2, 3, 4, 5);
// UNSAFE — AtomicInteger is thread-safe individually but the OVERALL C
// for stateful lambdas with parallel streams — prefer reduction/collec
AtomicInteger count = new AtomicInteger(0);
        small.parallelStream().forEach(n -> count.incrementAndGet());
// OK h
// NOT BENEFICIAL: very SMALL datasets — thread coordination overhead E
long startSmall = System.currentTimeMillis();
        small.parallelStream().mapToInt(Integer::intValue).sum();
System.out

In [139]:
result_docs = rag_retriever.retrieve("How we can implement load balancing")
result_docs
for res in result_docs:
    print(res['content'])

Reterieving documents for query: 'How we can implement load balancing'
Top k: 5, score threshold: 0.0
Generating embedding for 1 texts...
 Progress: 1/1 texts processed
✅ Generated embeddings with shape: (1, 4096)
Reterieved 5 documents (after filtering)
api.example.com -> resolves to us-east-1 ALB IP for US clients, eu-west-1 ALB
IP for EU clients
# Dedicated L7 load balancer (sits in the actual request path, per-request
decisions)
Client -> ALB (checks health, routes per-request) -> healthy backend instance
java
class WeightedRoundRobinBalancer {
record Server(String address, int weight) {}
private final List<Server> servers;
private final List<String> expandedPool = new ArrayList<>();
private int index = 0;
WeightedRoundRobinBalancer(List<Server> servers) {
this.servers = servers;
for (Server s : servers) {
for (int i = 0; i < s.weight(); i++) expandedPool.add(s.address());
}
Collections.shuffle(expandedPool);
// avoid clustering same-server pi
}
synchronized String next() {
String 

# Integration VectorDB context pipeline with LLM output

In [140]:
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [141]:
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")

In [142]:
llm = ChatOllama(
    model=OLLAMA_MODEL.split(":")[-1],
    temperature=0.1
)

In [143]:
## Simple RAG function: Retrieve context + generate response
def rag_sample(query, retriever, llm, top_k=3):
    ## Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relavent context found to answer the question"

    ## generate the answer using the llm
    prompt = f"""
        Use the following context to answer the question conciesly.
        context: {context}

        Question: {query}

        Answer:
        """

    response = llm.invoke([prompt])
    return response



In [144]:
answer = rag_sample("What is the difference between a Process and a Thread?", rag_retriever, llm)
print(answer.content)

Reterieving documents for query: 'What is the difference between a Process and a Thread?'
Top k: 3, score threshold: 0.0
Generating embedding for 1 texts...
 Progress: 1/1 texts processed
✅ Generated embeddings with shape: (1, 4096)
Reterieved 3 documents (after filtering)
A **Process** is an independent program with its own dedicated and isolated memory space (including heap, stack, code segment, and data segment).

A **Thread** is the smallest unit of execution within a process. While multiple threads within the same process share the process's memory (like the heap and static data), each thread maintains its own private resources, such as its own stack, program counter, and registers.

**In summary:** Processes are isolated units, while threads are concurrent workers that share resources within a single process.


In [145]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipline with extra feature:
    - Returns answer, sources, confidence score, and optionally full context
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            'answer': 'No relevent context found.',
            'source': [],
            'confidence': 0.0,
            'context': ''
            }

    # Prepare context and source
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', "unknown")),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300]+'...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""
        Use the following context to answer the question concisely.

        Context: {context}

        Question: {query}

        Answer: 
        """

    response = llm.invoke([prompt])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output


In [146]:
answer = rag_advanced("What is multithreading?", rag_retriever, llm, top_k=5, min_score=0, return_context=True)
print("Answer: ", answer['answer'])
print('Sources: ', answer['sources'])
print('Confidence: ', answer['confidence'])
print("Context Preview: ", answer['context'][:300])

Reterieving documents for query: 'What is multithreading?'
Top k: 5, score threshold: 0
Generating embedding for 1 texts...
 Progress: 1/1 texts processed
✅ Generated embeddings with shape: (1, 4096)
Reterieved 5 documents (after filtering)
Answer:  Multithreading is the ability of a CPU or a single program to run multiple threads concurrently, which can happen either truly in parallel (on multicore) or interleaved (on single core via context switching).
Sources:  [{'source': 'Java_Multithreading_Basic_To_Advanced.md.pdf', 'page': 2, 'score': 0.5617989003658295, 'preview': 'Q2. What is Multithreading, and what are its key advantages and disadvantages?\nMultithreading is the ability of a CPU or a single program to run multiple threads\nconcurrently, either truly in parallel (on multicore) or interleaved (on single core via\ncontext switching).\npublic class ProcessVsThreadD...'}, {'source': 'Java_Multithreading_Basic_To_Advanced.md.pdf', 'page': 1, 'score': 0.4129346013069153, 'preview'